# LoRA fine-tuning: bge-base-en-v1.5 (Kaggle GPU)

**Run this notebook on Kaggle, not locally.** Fine-tuning (even a small LoRA adapter) is
the kind of job that benefits from a real GPU and an uninterrupted run — Kaggle's free GPU
sessions (P100 16GB / 2xT4, 30 GPU-hrs/week) give us that at zero cost.

**What this notebook produces** (the Phase-4 fine-tuning exit-gate artifacts, per
`docs/ShopTalk_Plan.md` §2.3):
1. The LoRA-adapted encoder, trained via `src.embeddings.finetune.trainer.run()` on synthetic
   query/attribute-hard-negative triplets (eval-integrity gate enforced — no golden-set
   product or query leaks into training).
2. A before/after comparison table: **separation margin** (mean `cos(query, positive) -
   cos(query, negative)`) and **category clustering** (intra- vs inter-`product_type`
   cosine similarity), pretrained vs fine-tuned.
3. A zipped copy of the LoRA adapter weights to pull back to the Mac's `weights/finetuned/`.

**Setup on Kaggle (one-time, before running cells):**
1. Create a new Notebook, enable a GPU accelerator (Settings -> Accelerator -> **GPU T4 x2**,
   not P100 -- see the kernel-compatibility cell below for why).
2. Upload `data/processed/products_enriched.parquet` (or, if you haven't run captioning yet,
   `data/processed/products.parquet`) from this repo as a Kaggle **Dataset** (New Dataset ->
   drag the file in) and attach it to this notebook (Add Input -> your dataset). Either file
   works -- `doc_text` falls back gracefully when `visual_caption` is absent.
3. Run the cells top to bottom, **starting with the ABI-corruption check immediately below.**

> **If you ever run `pip install -r requirements.txt` here:** it downgrades `numpy`
> underneath Kaggle's pre-compiled `pandas`, permanently corrupting *this container's*
> `/usr/local/lib/python3.12/dist-packages` (`numpy.dtype size changed... Expected 96...
> got 88`). **`Run -> Restart session` will NOT fix it** -- that restarts the kernel
> *process*, not the container filesystem, and the corrupted files persist right where pip
> wrote them. The only real fix is to close this notebook and start a **brand-new
> session/copy** (a fresh container), then never run that install line again -- the setup
> cell below intentionally doesn't.

In [ ]:
# Run this cell FIRST, before anything else (even the clone). It eagerly forces
# `numpy.random` to load -- the exact submodule whose Cython-compiled .pyx raises
# "ValueError: numpy.dtype size changed... Expected 96... got 88" if numpy/pandas are
# C-ABI-mismatched. `import numpy` alone does NOT catch this: numpy 2.x lazily defers
# loading `numpy.random` via __getattr__, so the corruption hides until something forces it.
#
# If this raises: DO NOT just "Restart session" and re-run. Kaggle's restart restarts the
# *kernel process*, not the container filesystem -- and `pip install` writes into
# /usr/local/lib/python3.12/dist-packages, which is writable and PERSISTS across restarts.
# A prior `pip install -r requirements.txt` (which forces our pinned numpy onto Kaggle's
# pre-compiled pandas) leaves corrupted files there permanently for this container. The
# only fix is a genuinely fresh container: close this notebook and open a brand-new
# session/copy, then run cells top to bottom WITHOUT ever running `pip install -r
# requirements.txt` (the setup cell below deliberately never does this, for this reason).
try:
    import numpy as np

    np.random.default_rng()  # forces the lazy numpy.random import -- where the ABI check fires
    import pandas as pd

    print(f"numpy {np.__version__}  pandas {pd.__version__} -- imported cleanly, ABI consistent. Safe to proceed.")
except ValueError as e:
    if "numpy.dtype size changed" not in str(e):
        raise
    raise RuntimeError(
        "numpy/pandas C-ABI corruption detected in THIS CONTAINER "
        f"({e}). This is permanent for this session -- 'Restart session' restarts the "
        "kernel process but NOT /usr/local/lib/python3.12/dist-packages, where the "
        "corrupted files live. Close this notebook and start a brand-new session/copy "
        "(fresh container), then run cells top to bottom -- never run `pip install -r "
        "requirements.txt` here; the setup cell below intentionally doesn't."
    ) from e

In [ ]:
# Clone the repo (public).
#
# `rm -rf` first makes this cell idempotent: a half-finished clone from an earlier attempt
# would otherwise leave a stale `ShopTalk/` dir that `git clone` skips -- producing a
# confusing "ModuleNotFoundError: No module named 'src.embeddings.finetune'" if `src/`
# exists but is missing subpackages added after that stale clone was made.
!rm -rf ShopTalk
!git clone --depth 1 https://github.com/dceshubh/ShopTalk.git
%cd ShopTalk

# Deliberately DO NOT `pip install -r requirements.txt` here. Kaggle's base image ships
# numpy/pandas/pyarrow/torch as one mutually-compiled stack (built against each other's C
# ABI). Forcing our pinned versions on top downgrades numpy beneath Kaggle's pre-compiled
# pandas -> "ValueError: numpy.dtype size changed... Expected 96 ... got 88" (a binary
# incompatibility, not a Python-level version conflict).
#
# We only install what's actually missing (sentence-transformers + peft are NOT in
# Kaggle's base image), and with --no-deps so pip can't cascade-resolve its way into
# another ABI break -- Kaggle's torch/transformers are new enough for both.
import importlib
import subprocess
import sys

CHECKS = [
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("pyarrow", "pyarrow"),
    ("torch", "torch"),
    ("yaml", "pyyaml"),
    ("pydantic", "pydantic"),
    ("transformers", "transformers"),
    ("sentence_transformers", "sentence-transformers"),
    ("peft", "peft"),
]
for import_name, pip_name in CHECKS:
    try:
        mod = importlib.import_module(import_name)
        print(f"{import_name:20s} OK      {getattr(mod, '__version__', '')}")
    except ImportError:
        print(f"{import_name:20s} MISSING -- installing {pip_name} (--no-deps, to protect Kaggle's matched ABI stack)")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", pip_name], check=True)
        importlib.import_module(import_name)

In [ ]:
# Fail fast if the attached GPU's compute-capability architecture has no compiled CUDA
# kernels in this torch build -- "torch.AcceleratorError: CUDA error: no kernel image is
# available for execution on the device" (cudaErrorNoKernelImageForDevice). Kaggle's free
# accelerators include both P100 (Pascal, sm_60) and T4 (Turing, sm_75): modern PyTorch
# wheels increasingly ship without sm_60 kernel images to save binary size, while T4
# (sm_75) has the broadest official wheel coverage. Catching this here costs one
# GPU-second; finding it mid-training would burn through your free quota on a pure failure.
import torch

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    major, minor = torch.cuda.get_device_capability(0)
    arch_list = torch.cuda.get_arch_list()
    sm = f"sm_{major}{minor}"
    print(f"GPU: {name}  compute capability: {sm}")
    print(f"torch compiled kernel architectures: {arch_list}")
    if not any(sm in a for a in arch_list):
        raise RuntimeError(
            f"{name} is {sm}, but this torch build only has compiled kernels for {arch_list}. "
            "This is exactly the 'no kernel image is available for execution on the device' "
            "failure mode -- it will reproduce on every batch, not just some. "
            "Fix: Settings -> Accelerator -> switch to 'GPU T4 x2' (sm_75, the architecture "
            "with the broadest official PyTorch wheel support), then "
            "Run -> Restart session and re-run from the top."
        )
    print("GPU kernel coverage OK -- safe to proceed.")
else:
    raise RuntimeError(
        "No CUDA GPU detected -- check Settings -> Accelerator is set to a GPU option "
        "(GPU T4 x2). LoRA fine-tuning on CPU is impractically slow for this notebook."
    )

In [ ]:
# Locate the uploaded products parquet (attached as a Kaggle Dataset under /kaggle/input/)
# and copy it into the path src.common.config expects. Prefer products_enriched.parquet
# (has visual_caption) but fall back to products.parquet -- doc_text() handles a missing
# visual_caption column gracefully via Series.get.
import glob
import shutil
from pathlib import Path

Path("data/processed").mkdir(parents=True, exist_ok=True)

enriched = glob.glob("/kaggle/input/**/products_enriched.parquet", recursive=True)
plain = glob.glob("/kaggle/input/**/products.parquet", recursive=True)
candidates = enriched or plain
assert candidates, (
    "No products(_enriched).parquet found under /kaggle/input -- upload one as a Kaggle "
    "Dataset (New Dataset -> drag in data/processed/products(_enriched).parquet) and "
    "attach it via Add Input."
)
src_path = candidates[0]
dest_name = "products_enriched.parquet" if enriched else "products.parquet"
dest_path = f"data/processed/{dest_name}"
shutil.copy(src_path, dest_path)
print("Copied", src_path, "->", dest_path)

In [ ]:
# Build the training triplets up front -- this also runs the Phase-4 eval-integrity gate
# (assert_eval_integrity inside build_training_triplets): no golden-set product or query
# can leak into training, or this raises AssertionError before any GPU time is spent.
import pandas as pd

from src.common.config import load_config
from src.embeddings.finetune.triplet_mining import build_training_triplets
from src.eval.harness import load_golden_set

cfg = load_config()
products_path = dest_path  # from the previous cell
df = pd.read_parquet(products_path)
golden_cases = load_golden_set()
triplets = build_training_triplets(df, golden_cases)

print(f"products: {len(df)}  golden cases: {len(golden_cases)}  training triplets: {len(triplets)}")

In [ ]:
# Baseline (pretrained) metrics -- separation_margin and category_clustering computed
# directly from embeddings, no Chroma index needed. This is the "before" half of the
# fine-tuned-vs-pretrained comparison.
from src.embeddings.encode import load_encoder
from src.embeddings.finetune.metrics import category_clustering, separation_margin

base_model_name = cfg["finetune"]["base_model"]
pretrained_encoder = load_encoder(base_model_name)

pretrained_margin = separation_margin(pretrained_encoder, triplets, df)
pretrained_clustering = category_clustering(pretrained_encoder, df)

print(f"pretrained separation margin: {pretrained_margin:.4f}")
print(f"pretrained category clustering: {pretrained_clustering}")

In [ ]:
# LoRA fine-tune -- single-command entrypoint per docs/ShopTalk_Plan.md §2.3
# (`python -m src.embeddings.finetune.trainer`, here called as `run()` so we can capture
# the output path). Hyperparameters come from `finetune` in configs/config.yaml
# (epochs=3, batch_size=32, lr=2e-5, LoRA r=16/alpha=32/dropout=0.1).
from src.embeddings.finetune.trainer import run as run_finetune

output_dir = run_finetune(products_path=products_path, output_dir="/kaggle/working/finetuned-bge-lora-v1")
print("Saved fine-tuned encoder to", output_dir)

In [ ]:
# Fine-tuned metrics -- same triplets/df, same metric functions, just a different encoder.
# Loaded directly with SentenceTransformer (not load_encoder, which only accepts registered
# HF model names) but wrapped in the same Encoder dataclass so separation_margin/
# category_clustering see an identical interface -- and with the base model's prefix
# convention, since LoRA only adapts attention weights, not the input format.
from sentence_transformers import SentenceTransformer

from src.embeddings.encode import _PREFIX_CONVENTIONS, Encoder

query_prefix, passage_prefix = _PREFIX_CONVENTIONS[base_model_name]
finetuned_model = SentenceTransformer(str(output_dir), device=pretrained_encoder.device)
finetuned_encoder = Encoder(
    model_name=f"{base_model_name}-lora-v1",
    model=finetuned_model,
    device=pretrained_encoder.device,
    query_prefix=query_prefix,
    passage_prefix=passage_prefix,
)

finetuned_margin = separation_margin(finetuned_encoder, triplets, df)
finetuned_clustering = category_clustering(finetuned_encoder, df)

print(f"fine-tuned separation margin: {finetuned_margin:.4f}")
print(f"fine-tuned category clustering: {finetuned_clustering}")

In [ ]:
# Before/after comparison table -- the Phase-4 "fine-tuned beats pretrained" exit-gate
# artifact. Category clustering is reported as-is, not claimed to improve: attribute hard
# negatives can reduce intra-category compactness while still widening the separation
# margin (docs/ShopTalk_Plan.md §2.3).
comparison = pd.DataFrame(
    {
        "metric": [
            "separation_margin",
            "intra_category_mean_cosine",
            "inter_category_mean_cosine",
        ],
        "pretrained": [
            pretrained_margin,
            pretrained_clustering["intra_category_mean_cosine"],
            pretrained_clustering["inter_category_mean_cosine"],
        ],
        "finetuned": [
            finetuned_margin,
            finetuned_clustering["intra_category_mean_cosine"],
            finetuned_clustering["inter_category_mean_cosine"],
        ],
    }
)
comparison["delta"] = comparison["finetuned"] - comparison["pretrained"]
print(comparison.to_string(index=False))
comparison.to_csv("/kaggle/working/finetune_before_after.csv", index=False)

In [ ]:
# Zip the LoRA adapter weights for download -- this is what gets pulled back to the Mac's
# weights/finetuned/ directory (gitignored, not committed).
import shutil

zip_path = shutil.make_archive("/kaggle/working/finetuned-bge-lora-v1", "zip", root_dir=str(output_dir))
print("Zipped weights to", zip_path)

## Pulling results back to the Mac

From the Kaggle **Output** tab, download:
- `finetuned-bge-lora-v1.zip` -> unzip locally into
  `weights/finetuned/bge-base-en-v1.5-lora-v1/` (the path `src.embeddings.finetune.trainer.run()`
  would also produce locally, if you had a GPU to run it on)
- `finetune_before_after.csv` -> the separation-margin / category-clustering before/after
  numbers for the Phase-4 exit-gate writeup in `docs/ShopTalk_Plan.md`

Then, locally, point the retrieval encoder at `weights/finetuned/bge-base-en-v1.5-lora-v1`
(or re-run `python -m src.embeddings.finetune.trainer` directly if you ever get access to a
local GPU) and record the before/after numbers in the Phase-4 "Build notes" section.